In [ ]:
# pip install implicit

# implicit ALS (Hu & Koren 2008)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

SOURCE = "/content/drive/MyDrive/EV_charging_project/data_processing_files/processed_HCM_new.csv"      # change path if needed
TARGET = "top_active_users.csv"

df = pd.read_csv(SOURCE, low_memory=False)

top_users = (
    df["id_tag"]
    .value_counts()
    .nlargest(10)          # top frequent users
    .index
)

df_top = df[df["id_tag"].isin(top_users)].copy()

df_top.to_csv(TARGET, index=False)

print(f"Saved {len(df_top):,} rows for top n most-active users  {TARGET}")
print("\nSample:")
print(df_top.head())


Saved 2,966 rows for top n most-active users  top_active_users.csv

Sample:
      ma_tram         ma_tru          ma_cong  thoi_gian_sac_hours  \
5   C.HCM0192  C.HCM0192.002  C.HCM0192.002.1             0.215000   
9   C.HCM0192  C.HCM0192.004  C.HCM0192.004.2             0.806667   
10  C.HCM0192  C.HCM0192.003  C.HCM0192.003.2             0.964722   
13  C.HCM0192  C.HCM0192.004  C.HCM0192.004.2             0.328611   
14  C.HCM0192  C.HCM0192.004  C.HCM0192.004.1             0.715833   

    cong_suat_calculated  muc_pin_da_sac  cong_suat  dien_nang_tieu_thu  \
5              40.232558              22          0                8.65   
9              34.338843              62          0               27.70   
10             42.043190              42          0               40.56   
13             57.119188              43          0               18.77   
14             43.990687              73          0               31.49   

    muc_pin_bat_dau  muc_pin_ket_thuc  ... thoi_gian

In [ ]:
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/EV_charging_project/data_processing_files/processed_HCM_new.csv"

cols = ["thoi_gian_bat_dau", "so_khung"]            # add others if needed
df = pd.read_csv(CSV_PATH, usecols=cols, low_memory=False)

df["thoi_gian_bat_dau"] = pd.to_datetime(df["thoi_gian_bat_dau"],
                                         errors="coerce")
df = df.dropna(subset=["thoi_gian_bat_dau", "so_khung"])
df["hour"] = df["thoi_gian_bat_dau"].dt.hour

rush_hour = df["hour"].value_counts().idxmax()
rush_count = df["hour"].value_counts().max()
print(f"Rush hour: {rush_hour:02d}:00 with {rush_count:,} sessions")

rush_df = df[df["hour"] == rush_hour]
top10 = (rush_df["so_khung"]
         .value_counts()
         .head(10)
         .reset_index()
         .rename(columns={"index": "so_khung",
                          "so_khung": "sessions_in_hour"}))

print("\nTop-10 so_khung during rush hour:")
print(top10)

top10.to_csv("top10_users_rush_hour.csv", index=False)
print("\nSaved summary to 'top10_users_rush_hour.csv'")


Rush hour: 23:00 with 365 sessions

Top-10 so_khung during rush hour:
    sessions_in_hour  count
0  RLLV1AEBXPV013490     10
1  RLLV1AUA4PV706872     10
2  RLLV1AFA8NV890384     10
3  RLLV1ATB7PV709145      8
4  RLLV4HDH6RH705950      7
5  RLLVAG8C0RH704598      6
6  RLLV1AFAXPV890227      5
7  RLLV1AFA5NV890097      5
8  RLNV5JSE8RH713472      5
9  RLLV1ATB6PV711436      4

Saved summary to 'top10_users_rush_hour.csv'


In [ ]:
"""
Create Fisher-market valuation matrix V_ij from EV-charging logs that
contain these fields:

    ma_tram, ma_tru, ma_cong, thoi_gian_sac_hours, cong_suat_calculated,
    muc_pin_da_sac, cong_suat, dien_nang_tieu_thu, muc_pin_bat_dau,
    muc_pin_ket_thuc, so_khung, dong, dien_ap, thoi_gian_ket_thuc,
    id_tag, ma_giao_dich_tren_emsp, thoi_gian_bat_dau, thoi_gian_dung_sac,
    charger_id, charger_type, charger_power_kW, EV_model,
    EV_battery_capacity_kWh

Output:
    user_factors.npy, item_factors.npy      (for streaming valuations)
    user_lookup.csv,  charger_lookup.csv    (id_tag row, charger_id block)
    optional dense valuation_matrix.npy     (if SAVE_DENSE_V=True)
"""

import os, glob
from typing import List, Tuple

import numpy as np
import pandas as pd
import scipy.sparse as sp
from implicit.als import AlternatingLeastSquares
from tqdm.auto import tqdm


LOG_GLOB          = "top_active_users.csv"
TIME_COL          = "thoi_gian_bat_dau"
USER_COL          = "id_tag"
CHARGER_COL       = "charger_id"
SIGNAL_COL        = "dien_nang_tieu_thu"
SLOT_MINUTES      = 15
ALPHA             = 20.0
FACTORS           = 64
ITERATIONS        = 25
REGULARISATION    = 1e-4
USE_GPU           = False
SAVE_DENSE_V      = False
OUT_DIR           = "./valuation_out"
# --------------------------------------------------------------------------



def load_logs(pattern: str) -> pd.DataFrame:
    frames = []
    for fp in tqdm(sorted(glob.glob(pattern)), desc="Reading log files"):
        if fp.endswith(".parquet"):
            frames.append(pd.read_parquet(fp))
        else:
            frames.append(pd.read_csv(fp))
    if not frames:
        raise RuntimeError("No data files found for pattern: " + pattern)
    return pd.concat(frames, ignore_index=True)


def add_slot(df: pd.DataFrame, time_col: str, slot_minutes: int) -> None:
    ts = pd.to_datetime(df[time_col])
    df["slot"] = (ts.dt.hour * (60 // slot_minutes)
                  + ts.dt.minute // slot_minutes)


def encode_series(series: pd.Series) -> Tuple[np.ndarray, List[str]]:
    codes, uniques = pd.factorize(series)
    return codes.astype(np.int32), uniques.tolist()


def build_sparse(df: pd.DataFrame,
                 n_users: int,
                 n_items: int,
                 signal_col: str) -> sp.csr_matrix:
    """Return implicit-feedback matrix R (kWh or 1s) in CSR form."""
    data = df[signal_col].astype(np.float32).values
    rows = df["u"].values
    cols = df["item"].values
    return sp.coo_matrix((data, (rows, cols)),
                         shape=(n_users, n_items)).tocsr()


def fit_als(R: sp.csr_matrix, alpha: float, **als_kw):
    model = AlternatingLeastSquares(
        factors=als_kw.get("factors", FACTORS),
        regularization=als_kw.get("reg", REGULARISATION),
        iterations=als_kw.get("iters", ITERATIONS),
        use_gpu=als_kw.get("gpu", USE_GPU),
        calculate_training_loss=False)
    model.fit(R * alpha)
    return model


def save_artifacts(model,
                   user_labels, charger_labels,
                   n_slots, save_dense, out_dir):

    os.makedirs(out_dir, exist_ok=True)

    model.user_factors  = np.maximum(model.user_factors,  0.0)
    model.item_factors  = np.maximum(model.item_factors,  0.0)

    row_max = model.user_factors @ model.item_factors.T
    row_max = row_max.max(axis=1, keepdims=True) + 1e-9
    scale   = 10.0 / row_max           # shape (n_users,1)
    model.user_factors *= scale

    np.save(f"{out_dir}/user_factors.npy",  model.user_factors.astype(np.float32))
    np.save(f"{out_dir}/item_factors.npy",  model.item_factors.astype(np.float32))

    pd.Series(user_labels).to_csv(f"{out_dir}/user_lookup.csv",
                                  index=False, header=["id_tag"])
    pd.Series(charger_labels).to_csv(f"{out_dir}/charger_lookup.csv",
                                     index=False, header=["charger_id"])

    if save_dense:
        V = model.user_factors @ model.item_factors.T
        V = np.clip(V, 1e-3, None)            # strictly positive
        np.save(f"{out_dir}/valuation_matrix.npy", V.astype(np.float32))
        print("Dense valuation matrix saved – shape", V.shape)


def main():
    print("Loading data …")
    df = load_logs(LOG_GLOB)

    add_slot(df, TIME_COL, SLOT_MINUTES)
    n_slots = 60 // SLOT_MINUTES          # 96 for 15-min

    df["u"], user_labels      = encode_series(df[USER_COL])
    df["c"], charger_labels   = encode_series(df[CHARGER_COL])

    df["item"] = df["c"] * n_slots + df["slot"]
    n_users    = len(user_labels)
    n_items    = df["item"].max() + 1

    print(f"Users  : {n_users:,}")
    print(f"Chargers: {len(charger_labels):,}")
    print(f"Items  : {n_items:,}  (chargers × slots)")

    R = build_sparse(df, n_users, n_items, SIGNAL_COL)
    print(f"Sparse matrix nnz = {R.nnz:,}")

    print("Training implicit-ALS …")
    model = fit_als(R, alpha=ALPHA)

    save_artifacts(model, user_labels, charger_labels,
                   n_slots, SAVE_DENSE_V, OUT_DIR)

    print("✓ All done – artifacts in", OUT_DIR)


if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'implicit'

# RELU

In [ ]:
import os, glob
import numpy as np, pandas as pd, scipy.sparse as sp
from implicit.als import AlternatingLeastSquares
from typing import List, Tuple
from tqdm.auto import tqdm

LOG_GLOB     = "/content/drive/MyDrive/EV_charging_project/data_processing_files/best_HCM_station.csv"
TIME_COL     = "thoi_gian_bat_dau"
USER_COL     = "so_khung"
CHARGER_COL  = "charger_id"
SIGNAL_COL   = "dien_nang_tieu_thu"
SLOT_MINUTES = 15
ALPHA        = 20.0
FACTORS      = 32
ITERATIONS   = 15
REGULARISATION = 1e-3
OUT_DIR      = "./valuation_out_23h"
SAVE_DENSE_V = False

def load_logs(pattern: str) -> pd.DataFrame:
    frames = []
    for fp in tqdm(glob.glob(pattern), desc="Reading logs"):
        frames.append(pd.read_csv(fp, low_memory=False))
    if not frames:
        raise RuntimeError("No files match pattern " + pattern)
    return pd.concat(frames, ignore_index=True)

def add_slot(df: pd.DataFrame, tcol: str):
    t = pd.to_datetime(df[tcol], errors="coerce")
    df["hour"] = t.dt.hour
    df["slot"] = t.dt.minute // SLOT_MINUTES

def encode(series: pd.Series):
    codes, uniq = pd.factorize(series)
    return codes.astype(np.int32), uniq.tolist()

def build_sparse(df: pd.DataFrame, n_u: int, n_i: int) -> sp.csr_matrix:
    data = df[SIGNAL_COL].astype(np.float32).values
    return sp.coo_matrix((data, (df["u"], df["item"])),
                         shape=(n_u, n_i)).tocsr()

def save(model, users, chargers):
    os.makedirs(OUT_DIR, exist_ok=True)
    # clamp & scale
    model.user_factors = np.maximum(model.user_factors, 0)
    model.item_factors = np.maximum(model.item_factors, 0)
    row_max = (model.user_factors @ model.item_factors.T).max(axis=1, keepdims=True) + 1e-12
    model.user_factors *= 10 / row_max
    np.save(f"{OUT_DIR}/user_factors.npy", model.user_factors.astype(np.float32))
    np.save(f"{OUT_DIR}/item_factors.npy", model.item_factors.astype(np.float32))
    pd.Series(users).to_csv(f"{OUT_DIR}/user_lookup.csv", index=False, header=[USER_COL])
    pd.Series(chargers).to_csv(f"{OUT_DIR}/charger_lookup.csv", index=False, header=[CHARGER_COL])

def main():
    df = load_logs(LOG_GLOB)
    df["charger_id"] = df["ma_tru"]

    add_slot(df, TIME_COL)
    df = df[df["hour"] == 23]

    top10 = (df[USER_COL].value_counts().head(10).index)
    df = df[df[USER_COL].isin(top10)]

    df["u"], user_labels     = encode(df[USER_COL])
    df["c"], charger_labels  = encode(df[CHARGER_COL])

    n_users   = len(user_labels)           # = 10
    n_chg     = len(charger_labels)
    n_slots = 60 // SLOT_MINUTES          # = 4 slots
    df["item"] = (df["c"] * n_slots + df["slot"]).astype(int)

    n_items = int(df["item"].max()) + 1


    print(f"Users    : {n_users}")
    print(f"Chargers : {n_chg}")
    print(f"Goods    : {n_items}  (chargers × 4 slots)")

    R = build_sparse(df, n_users, n_items)
    model = AlternatingLeastSquares(factors=FACTORS,
                                    regularization=REGULARISATION,
                                    iterations=ITERATIONS,
                                    use_gpu=False)
    model.fit(R * ALPHA)

    save(model, user_labels, charger_labels)
    print("✓ valuations for top-10 users during 23:00–23:59 saved to", OUT_DIR)

if __name__ == "__main__":
    main()


Reading logs:   0%|          | 0/1 [00:00<?, ?it/s]

Users    : 10
Chargers : 13
Goods    : 51  (chargers × 4 slots)


  0%|          | 0/15 [00:00<?, ?it/s]

✓ valuations for top-10 users during 23:00–23:59 saved to ./valuation_out_23h


In [ ]:
import numpy as np

U = np.load("valuation_out_23h/user_factors.npy")
P = np.load("valuation_out_23h/item_factors.npy")

V = U @ P.T

def v_for_user(i):
    return U[i] @ P.T


In [ ]:
V.shape

# Softmax

In [ ]:
"""
Rush-hour (23:00–23:59) valuation matrix for the 10 most-active
`so_khung` users.  Goods = charger × 15-min slot actually used.

Valuation construction
  • implicit ALS    raw logits  L = U Pᵀ
  • sigmoid on factors          (positivity, bounded)
  • row-softmax on logits       (rows sum to 10)

Outputs (in OUT_DIR)
  user_factors.npy        ┐
  item_factors.npy        │ post-sigmoid ≥0 factors
  user_lookup.csv         │
  charger_lookup.csv      │
  valuation_matrix.npy    ┘ dense V  (always saved, rows sum to 10)
"""
import os, glob
import numpy as np, pandas as pd, scipy.sparse as sp
from implicit.als import AlternatingLeastSquares
from tqdm.auto import tqdm

LOG_GLOB        = "/content/drive/MyDrive/EV_charging_project/data_processing_files/processed_HCM_new.csv"
TIME_COL        = "thoi_gian_bat_dau"
USER_COL        = "so_khung"
CHARGER_COL     = "charger_id"
SIGNAL_COL      = "dien_nang_tieu_thu"
SLOT_MINUTES    = 15
ALPHA           = 20.0
FACTORS         = 32
ITERATIONS      = 15
REG             = 1e-3
OUT_DIR         = "./valuation_out_23h_sigmoid"

def load_logs(pattern: str) -> pd.DataFrame:
    parts = [pd.read_csv(p, low_memory=False) for p in glob.glob(pattern)]
    if not parts:
        raise RuntimeError(f"No files match {pattern}")
    return pd.concat(parts, ignore_index=True)

def add_slot(df: pd.DataFrame):
    ts = pd.to_datetime(df[TIME_COL], errors="coerce")
    df["hour"] = ts.dt.hour
    df["slot"] = ts.dt.minute // SLOT_MINUTES

def code(series: pd.Series):
    c, uniq = pd.factorize(series)
    return c.astype(np.int32), uniq.tolist()

def sparse_R(df: pd.DataFrame, n_u: int, n_i: int) -> sp.csr_matrix:
    data = df[SIGNAL_COL].astype(np.float32).values
    return sp.coo_matrix((data, (df["u"], df["item"])),
                         shape=(n_u, n_i)).tocsr()

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def main():
    print("Loading logs …")
    df = load_logs(LOG_GLOB)
    df["charger_id"] = df["ma_tru"]

    add_slot(df)
    df = df[df["hour"] == 23]

    top10 = df[USER_COL].value_counts().head(10).index
    df = df[df[USER_COL].isin(top10)]

    df["u"], users      = code(df[USER_COL])
    df["c"], chargers   = code(df[CHARGER_COL])

    n_users  = len(users)
    n_slots  = 60 // SLOT_MINUTES      # 4
    df["item"] = (df["c"] * n_slots + df["slot"]).astype(int)
    n_items  = df["item"].max() + 1

    print(f"Users={n_users}, Goods={n_items}")

    R = sparse_R(df, n_users, n_items)
    als = AlternatingLeastSquares(factors=FACTORS,
                                  regularization=REG,
                                  iterations=ITERATIONS,
                                  use_gpu=False)
    als.fit(R * ALPHA)

    U_pos = sigmoid(als.user_factors)
    P_pos = sigmoid(als.item_factors)

    logits = U_pos @ P_pos.T
    logits -= logits.max(axis=1, keepdims=True)
    expV   = np.exp(logits)
    V      = 10.0 * expV / expV.sum(axis=1, keepdims=True)  # rows sum to 10

    os.makedirs(OUT_DIR, exist_ok=True)
    np.save(f"{OUT_DIR}/user_factors.npy",  U_pos.astype(np.float32))
    np.save(f"{OUT_DIR}/item_factors.npy",  P_pos.astype(np.float32))
    pd.Series(users).to_csv(f"{OUT_DIR}/user_lookup.csv",
                            index=False, header=[USER_COL])
    pd.Series(chargers).to_csv(f"{OUT_DIR}/charger_lookup.csv",
                               index=False, header=[CHARGER_COL])
    np.save(f"{OUT_DIR}/valuation_matrix.npy", V.astype(np.float32))

    print("✓ Saved rush-hour valuations to", OUT_DIR,
          f"(matrix shape {V.shape})")

if __name__ == "__main__":
    main()


Loading logs …
Users=10, Goods=22


  0%|          | 0/15 [00:00<?, ?it/s]

✓ Saved rush-hour valuations to ./valuation_out_23h_sigmoid (matrix shape (10, 22))


# Softmax full

In [ ]:
"""
Rush-hour (23:00–23:59) valuation matrix for the 10 most-active
`so_khung` users.  Goods = charger × 15-min slot actually used.

Valuation construction
  • implicit ALS    raw logits  L = U Pᵀ
  • sigmoid on factors          (positivity, bounded)
  • row-softmax on logits       (rows sum to 10)

Outputs (in OUT_DIR)
  user_factors.npy        ┐
  item_factors.npy        │ post-sigmoid ≥0 factors
  user_lookup.csv         │
  charger_lookup.csv      │
  valuation_matrix.npy    ┘ dense V  (always saved, rows sum to 10)
"""
import os, glob
import numpy as np, pandas as pd, scipy.sparse as sp
from implicit.als import AlternatingLeastSquares
from tqdm.auto import tqdm

LOG_GLOB        = "/content/drive/MyDrive/EV_charging_project/data_processing_files/processed_HCM_new.csv"
TIME_COL        = "thoi_gian_bat_dau"
USER_COL        = "so_khung"
CHARGER_COL     = "charger_id"
SIGNAL_COL      = "dien_nang_tieu_thu"
SLOT_MINUTES    = 15
ALPHA           = 20.0
FACTORS         = 32
ITERATIONS      = 15
REG             = 1e-3
OUT_DIR         = "./valuation_out_23h_sigmoid"

def load_logs(pattern: str) -> pd.DataFrame:
    parts = [pd.read_csv(p, low_memory=False) for p in glob.glob(pattern)]
    if not parts:
        raise RuntimeError(f"No files match {pattern}")
    return pd.concat(parts, ignore_index=True)

def add_slot(df: pd.DataFrame):
    ts = pd.to_datetime(df[TIME_COL], errors="coerce")
    df["hour"] = ts.dt.hour
    df["slot"] = ts.dt.minute // SLOT_MINUTES

def code(series: pd.Series):
    c, uniq = pd.factorize(series)
    return c.astype(np.int32), uniq.tolist()

def sparse_R(df: pd.DataFrame, n_u: int, n_i: int) -> sp.csr_matrix:
    data = df[SIGNAL_COL].astype(np.float32).values
    return sp.coo_matrix((data, (df["u"], df["item"])),
                         shape=(n_u, n_i)).tocsr()

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def main():
    print("Loading logs …")
    df = load_logs(LOG_GLOB)
    df["charger_id"] = df["ma_tru"]

    add_slot(df)
    df = df[df["hour"] == 23]

    top10 = df[USER_COL].value_counts().head(10).index
    df = df[df[USER_COL].isin(top10)]

    df["u"], users      = code(df[USER_COL])
    df["c"], chargers   = code(df[CHARGER_COL])

    n_users  = len(users)
    n_slots  = 60 // SLOT_MINUTES      # 4
    df["item"] = (df["c"] * n_slots + df["slot"]).astype(int)
    n_items  = df["item"].max() + 1

    print(f"Users={n_users}, Goods={n_items}")

    R = sparse_R(df, n_users, n_items)
    als = AlternatingLeastSquares(factors=FACTORS,
                                  regularization=REG,
                                  iterations=ITERATIONS,
                                  use_gpu=False)
    als.fit(R * ALPHA)

    U_pos = sigmoid(als.user_factors)
    P_pos = sigmoid(als.item_factors)

    logits = U_pos @ P_pos.T
    logits -= logits.max(axis=1, keepdims=True)
    expV   = np.exp(logits)
    V      = 10.0 * expV / expV.sum(axis=1, keepdims=True)  # rows sum to 10

    observed_goods = np.unique(df["item"])
    V_obs = V[:, observed_goods]
    P_pos_obs = P_pos[observed_goods]
    chargers_raw = chargers

    V_full  = np.zeros((n_users, n_items), dtype=np.float32)
    V_full[:, observed_goods] = V_obs

    P_pos_full = np.zeros((n_items, FACTORS), dtype=np.float32)
    P_pos_full[observed_goods] = P_pos_obs


    os.makedirs(OUT_DIR, exist_ok=True)
    np.save(f"{OUT_DIR}/user_factors.npy",  U_pos.astype(np.float32))
    np.save(f"{OUT_DIR}/item_factors.npy",  P_pos_full.astype(np.float32))
    pd.Series(users).to_csv(f"{OUT_DIR}/user_lookup.csv",
                            index=False, header=[USER_COL])
    pd.Series(chargers_raw).to_csv(f"{OUT_DIR}/charger_lookup.csv",
                                index=False, header=[CHARGER_COL])
    np.save(f"{OUT_DIR}/valuation_matrix.npy", V_full.astype(np.float32))

    print("✓ Saved rush-hour valuations to", OUT_DIR,
        f"(matrix shape {V_full.shape})")

if __name__ == "__main__":
    main()


#observed_goods = np.unique(df["item"])
#V_obs = V[:, observed_goods]
#P_pos_obs = P_pos[observed_goods]
#chargers_raw = chargers

#V_full  = np.zeros((n_users, n_items), dtype=np.float32)
#V_full[:, observed_goods] = V_obs

#P_pos_full = np.zeros((n_items, FACTORS), dtype=np.float32)
#P_pos_full[observed_goods] = P_pos_obs


Loading logs …
Users=10, Goods=22


  0%|          | 0/15 [00:00<?, ?it/s]

✓ Saved rush-hour valuations to ./valuation_out_23h_sigmoid (matrix shape (10, 22))


In [ ]:
import numpy as np

U = np.load("valuation_out_23h_sigmoid/user_factors.npy")
P = np.load("valuation_out_23h_sigmoid/item_factors.npy")

V = U @ P.T

def v_for_user(i):
    return U[i] @ P.T

FileNotFoundError: [Errno 2] No such file or directory: 'valuation_out_23h_sigmoid/user_factors.npy'

In [ ]:
"""
Rush-hour (23:00–23:59) valuation matrix for the 10 most-active
`so_khung` users.  Goods = EVERY charger × 15-min slot combination.
If a (charger, slot) never appears in the log, its valuation is 0.

Valuation construction
  • implicit ALS    raw logits  L = U Pᵀ   (only on observed goods)
  • sigmoid on factors          (positivity, bounded)
  • row-softmax on logits       (rows sum to 10)

Outputs (in OUT_DIR)
  user_factors.npy        ┐  same shape as ALS (10 × FACTORS)
  item_factors.npy        │  full ( C·T × FACTORS ), 0 rows for unseen goods
  user_lookup.csv         │
  charger_lookup.csv      │
  valuation_matrix.npy    ┘  dense V  (10 × C·T, rows sum to 10)
"""
import os, glob
import numpy as np, pandas as pd, scipy.sparse as sp
from implicit.als import AlternatingLeastSquares
from tqdm.auto import tqdm

LOG_GLOB   = "/content/drive/MyDrive/EV_charging_project/data_processing_files/processed_HCM_new.csv"
TIME_COL   = "thoi_gian_bat_dau"
USER_COL   = "so_khung"
CHARGER_COL= "charger_id"
SIGNAL_COL = "dien_nang_tieu_thu"
SLOT_MIN   = 15                      # minutes
ALPHA      = 20.0
FACTORS    = 32
ITERATIONS = 15
REG        = 1e-3
OUT_DIR    = "./valuation_out_23h_sigmoid_full"


def load_logs(pattern: str) -> pd.DataFrame:
    parts = [pd.read_csv(p, low_memory=False) for p in glob.glob(pattern)]
    if not parts:
        raise RuntimeError(f"No files match {pattern}")
    return pd.concat(parts, ignore_index=True)

def add_slot(df: pd.DataFrame):
    ts = pd.to_datetime(df[TIME_COL], errors="coerce")
    df["hour"] = ts.dt.hour
    df["slot"] = ts.dt.minute // SLOT_MIN

def code(series: pd.Series):
    c, uniq = pd.factorize(series)
    return c.astype(np.int32), uniq.tolist()

def sparse_R(df: pd.DataFrame, n_u: int, n_i_obs: int) -> sp.csr_matrix:
    data = df[SIGNAL_COL].astype(np.float32).values
    return sp.coo_matrix((data, (df["u"], df["item_obs"])),
                         shape=(n_u, n_i_obs)).tocsr()

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def main():
    print("Loading logs …")
    df = load_logs(LOG_GLOB)
    df["charger_id"] = df[""]

    add_slot(df)
    df = df[df["hour"] == 23]

    top10 = df[USER_COL].value_counts().head(10).index
    df = df[df[USER_COL].isin(top10)]

    df["u"], users        = code(df[USER_COL])
    df["c"], chargers_raw = code(df[CHARGER_COL])

    n_users   = len(users)
    n_charger = len(chargers_raw)
    n_slots   = 60 // SLOT_MIN
    n_items   = n_charger * n_slots

    df["item_full"] = (df["c"] * n_slots + df["slot"]).astype(int)
    observed_goods  = np.sort(df["item_full"].unique())
    obs2compact     = {g: i for i, g in enumerate(observed_goods)}
    df["item_obs"]  = df["item_full"].map(obs2compact)

    print(f"Users={n_users}, Chargers={n_charger}, Slots={n_slots}, "
          f"Observed goods={len(observed_goods)}, Full goods={n_items}")

    R_obs = sparse_R(df, n_users, len(observed_goods))
    als   = AlternatingLeastSquares(factors=FACTORS,
                                    regularization=REG,
                                    iterations=ITERATIONS,
                                    use_gpu=False)
    als.fit(R_obs * ALPHA)

    U_pos = sigmoid(als.user_factors)
    P_pos_obs = sigmoid(als.item_factors)

    logits_obs = U_pos @ P_pos_obs.T
    logits_obs -= logits_obs.max(axis=1, keepdims=True)
    expV_obs = np.exp(logits_obs)
    V_obs    = 10.0 * expV_obs / expV_obs.sum(axis=1, keepdims=True)

    V_full  = np.zeros((n_users, n_items), dtype=np.float32)
    V_full[:, observed_goods] = V_obs

    P_pos_full = np.zeros((n_items, FACTORS), dtype=np.float32)
    P_pos_full[observed_goods] = P_pos_obs

    os.makedirs(OUT_DIR, exist_ok=True)
    np.save(f"{OUT_DIR}/user_factors.npy",  U_pos.astype(np.float32))
    np.save(f"{OUT_DIR}/item_factors.npy",  P_pos_full.astype(np.float32))
    pd.Series(users).to_csv(f"{OUT_DIR}/user_lookup.csv",
                            index=False, header=[USER_COL])
    pd.Series(chargers_raw).to_csv(f"{OUT_DIR}/charger_lookup.csv",
                               index=False, header=[CHARGER_COL])
    np.save(f"{OUT_DIR}/valuation_matrix.npy", V_full.astype(np.float32))

    print("✓ Saved rush-hour valuations to", OUT_DIR,
          f"(matrix shape {V_full.shape})")

# --------------------------------------------------------------------------
if __name__ == "__main__":
    main()
